# Cross-Domain Evaluation (Full Coverage)
**Train on source A, test on source B's test split**

Evaluates **all 8 model families** across 6 cross-domain pairs = **48 evaluations**.

Models:
- TF-IDF + LR, TF-IDF + SVC (classical)
- CNN, LSTM, GRU (DL)
- DistilBERT, BERT, RoBERTa (transformer)

## Setup (Google Drive)
Drive root: `My Drive/PATH/TO`
- `data/splits/` — 21 split CSVs
- `models/{8 model families}/{twitter,skytrax,airlinequality}/` — single-source models

Results saved to `results/cross_domain.csv`.

**Runtime:** T4 ~45 min, L4/H100 ~15 min.

## 1. Setup

In [ ]:
!pip install -q transformers

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/PATH/TO"  # TODO: set to your Google Drive project path
SPLITS_DIR = f"{DRIVE_BASE}/data/splits"
MODELS_DIR = f"{DRIVE_BASE}/models"
RESULTS_DIR = f"{DRIVE_BASE}/results"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.isdir(SPLITS_DIR), f"Missing: {SPLITS_DIR}"
print("Paths OK.")

In [ ]:
import csv
import pickle
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_NAMES = ["negative", "neutral", "positive"]
SINGLE_SOURCES = ["twitter", "skytrax", "airlinequality"]
MAX_LENGTH_TWEET = 128
MAX_LENGTH_REVIEW = 512
DL_MAX_LEN_TWEET = 64
DL_MAX_LEN_REVIEW = 256
BATCH_SIZE = 32
PAD_IDX, UNK_IDX = 0, 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. DL Model Definitions (needed to reload checkpoints)

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, num_classes=3, num_filters=128, kernel_sizes=(3,4,5), dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        outs = [torch.relu(c(x)).max(dim=2).values for c in self.convs]
        return self.fc(self.dropout(torch.cat(outs, dim=1)))


class TextLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, hidden_dim=128, num_classes=3, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim*2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        return self.fc(self.dropout(torch.cat((h[0], h[1]), dim=1)))


class TextGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, hidden_dim=128, num_classes=3, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim*2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.embedding(x)
        _, h = self.gru(x)
        return self.fc(self.dropout(torch.cat((h[0], h[1]), dim=1)))

DL_CLASSES = {"cnn": TextCNN, "lstm": TextLSTM, "gru": TextGRU}
print("DL models defined.")

## 3. Utilities

In [ ]:
def load_test_split(source):
    path = os.path.join(SPLITS_DIR, f"{source}_test.csv")
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            texts.append(row["text"])
            labels.append(LABEL_MAP[row["label"]])
    return texts, labels


def compute_metrics(y_true, y_pred):
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }
    p = precision_score(y_true, y_pred, average=None, zero_division=0)
    r = recall_score(y_true, y_pred, average=None, zero_division=0)
    f = f1_score(y_true, y_pred, average=None, zero_division=0)
    for i, name in enumerate(LABEL_NAMES):
        m[f"{name}_precision"] = p[i]
        m[f"{name}_recall"] = r[i]
        m[f"{name}_f1"] = f[i]
    return m


def texts_to_seq(texts, word2idx, max_len):
    seq = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, t in enumerate(texts):
        toks = t.split()[:max_len]
        for j, tok in enumerate(toks):
            seq[i, j] = word2idx.get(tok, UNK_IDX)
    return seq


class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length",
                             max_length=self.max_length, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

print("Utilities loaded.")

## 4. Evaluators (one per model family)

In [ ]:
def eval_tfidf(model_key, train_src, test_src):
    d = f"{MODELS_DIR}/{model_key}/{train_src}"
    with open(f"{d}/tfidf.pkl", "rb") as f:
        tfidf = pickle.load(f)
    with open(f"{d}/model.pkl", "rb") as f:
        clf = pickle.load(f)
    texts, labels = load_test_split(test_src)
    return np.array(labels), clf.predict(tfidf.transform(texts))


def eval_dl(model_key, train_src, test_src):
    ckpt_path = f"{MODELS_DIR}/{model_key}/{train_src}/model.pt"
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    word2idx = ckpt["word2idx"]
    vocab_size = ckpt["vocab_size"]
    # Use the max_len the model was TRAINED on (stored in checkpoint)
    train_max_len = ckpt["max_len"]

    model = DL_CLASSES[model_key](vocab_size).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    texts, labels = load_test_split(test_src)
    X = texts_to_seq(texts, word2idx, train_max_len)
    loader = DataLoader(TensorDataset(torch.from_numpy(X), torch.tensor(labels)), batch_size=BATCH_SIZE)

    preds, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            ys.extend(yb.numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.array(ys), np.array(preds)


def eval_transformer(model_key, train_src, test_src):
    d = f"{MODELS_DIR}/{model_key}/{train_src}"
    tokenizer = AutoTokenizer.from_pretrained(d)
    model = AutoModelForSequenceClassification.from_pretrained(d).to(device)
    model.eval()
    texts, labels = load_test_split(test_src)
    max_len = MAX_LENGTH_TWEET if test_src == "twitter" else MAX_LENGTH_REVIEW
    ds = TransformerDataset(texts, labels, tokenizer, max_len)
    loader = DataLoader(ds, batch_size=BATCH_SIZE)
    preds, ys = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            att = batch["attention_mask"].to(device)
            logits = model(input_ids=ids, attention_mask=att).logits
            preds.extend(logits.argmax(1).cpu().numpy())
            ys.extend(batch["label"].numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.array(ys), np.array(preds)


MODELS = [
    ("tfidf_lr",   "tfidf"),
    ("tfidf_svc",  "tfidf"),
    ("cnn",        "dl"),
    ("lstm",       "dl"),
    ("gru",        "dl"),
    ("distilbert", "transformer"),
    ("bert",       "transformer"),
    ("roberta",    "transformer"),
]
EVALUATORS = {"tfidf": eval_tfidf, "dl": eval_dl, "transformer": eval_transformer}
print(f"Will evaluate {len(MODELS)} models x 6 pairs = {len(MODELS)*6} combinations")

## 5. Run All Cross-Domain Evaluations (crash-safe, resumable)

In [ ]:
OUT_PATH = f"{RESULTS_DIR}/cross_domain.csv"

# Load existing results for crash recovery
completed = set()
results = []
if os.path.exists(OUT_PATH):
    with open(OUT_PATH, encoding="utf-8") as f:
        results = list(csv.DictReader(f))
        completed = {(r["model"], r["train_source"], r["test_source"]) for r in results}
print(f"Already completed: {len(completed)}")

total = len(MODELS) * 6
n = len(completed)

for model_key, kind in MODELS:
    for train_src in SINGLE_SOURCES:
        for test_src in SINGLE_SOURCES:
            if train_src == test_src: continue
            if (model_key, train_src, test_src) in completed:
                print(f"[SKIP] {model_key} | {train_src} -> {test_src}")
                continue

            print(f"\n[EVAL {n+1}/{total}] {model_key} | train={train_src} -> test={test_src}")
            y_true, y_pred = EVALUATORS[kind](model_key, train_src, test_src)
            metrics = compute_metrics(y_true, y_pred)
            cm = confusion_matrix(y_true, y_pred)
            print(f"  Accuracy: {metrics['accuracy']:.4f} | Macro-F1: {metrics['macro_f1']:.4f}")
            print(f"  CM:\n{cm}")

            results.append({
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "model": model_key,
                "train_source": train_src,
                "test_source": test_src,
                **metrics,
            })
            n += 1

            with open(OUT_PATH, "w", encoding="utf-8", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
                w.writeheader()
                w.writerows(results)

print(f"\nDONE: {len(results)} total evaluations in {OUT_PATH}")

## 6. Summary Table

In [ ]:
print(f"{'Model':<12} {'Train':<12} {'Test':<12} {'Accuracy':>10} {'Macro-F1':>10}")
print("=" * 62)
for r in results:
    print(f"{r['model']:<12} {r['train_source']:<12} {r['test_source']:<12} {float(r['accuracy']):>10.4f} {float(r['macro_f1']):>10.4f}")

# Family-level summary: average cross-domain Macro-F1 per model
print("\n--- Average Cross-Domain Macro-F1 per Model ---")
from collections import defaultdict
agg = defaultdict(list)
for r in results:
    agg[r["model"]].append(float(r["macro_f1"]))
for m, scores in sorted(agg.items(), key=lambda x: -np.mean(x[1])):
    print(f"  {m:<12} mean={np.mean(scores):.4f} min={min(scores):.4f} max={max(scores):.4f}")

print(f"\nResults on Drive: {OUT_PATH}")